# Module 08: AES S-box Leakage & DPA/CPA Attacks — Lab

This lab performs a full CPA attack on AES, recovering key bytes one at a time.

**Objectives:**
1. Understand the S-box leakage model
2. Implement CPA attack on a single key byte
3. Visualize correlation traces for all key guesses
4. Extend to full 128-bit key recovery

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

print("=" * 60)
print("AES S-BOX LEAKAGE MODEL")
print("=" * 60)

# AES S-box (complete)
AES_SBOX = [
    0x63,0x7C,0x77,0x7B,0xF2,0x6B,0x6F,0xC5,0x30,0x01,0x67,0x2B,0xFE,0xD7,0xAB,0x76,
    0xCA,0x82,0xC9,0x7D,0xFA,0x59,0x47,0xF0,0xAD,0xD4,0xA2,0xAF,0x9C,0xA4,0x72,0xC0,
    0xB7,0xFD,0x93,0x26,0x36,0x3F,0xF7,0xCC,0x34,0xA5,0xE5,0xF1,0x71,0xD8,0x31,0x15,
    0x04,0xC7,0x23,0xC3,0x18,0x96,0x05,0x9A,0x07,0x12,0x80,0xE2,0xEB,0x27,0xB2,0x75,
    0x09,0x83,0x2C,0x1A,0x1B,0x6E,0x5A,0xA0,0x52,0x3B,0xD6,0xB3,0x29,0xE3,0x2F,0x84,
    0x53,0xD1,0x00,0xED,0x20,0xFC,0xB1,0x5B,0x6A,0xCB,0xBE,0x39,0x4A,0x4C,0x58,0xCF,
    0xD0,0xEF,0xAA,0xFB,0x43,0x4D,0x33,0x85,0x45,0xF9,0x02,0x7F,0x50,0x3C,0x9F,0xA8,
    0x51,0xA3,0x40,0x8F,0x92,0x9D,0x38,0xF5,0xBC,0xB6,0xDA,0x21,0x10,0xFF,0xF3,0xD2,
    0xCD,0x0C,0x13,0xEC,0x5F,0x97,0x44,0x17,0xC4,0xA7,0x7E,0x3D,0x64,0x5D,0x19,0x73,
    0x60,0x81,0x4F,0xDC,0x22,0x2A,0x90,0x88,0x46,0xEE,0xB8,0x14,0xDE,0x5E,0x0B,0xDB,
    0xE0,0x32,0x3A,0x0A,0x49,0x06,0x24,0x5C,0xC2,0xD3,0xAC,0x62,0x91,0x95,0xE4,0x79,
    0xE7,0xC8,0x37,0x6D,0x8D,0xD5,0x4E,0xA9,0x6C,0x56,0xF4,0xEA,0x65,0x7A,0xAE,0x08,
    0xBA,0x78,0x25,0x2E,0x1C,0xA6,0xB4,0xC6,0xE8,0xDD,0x74,0x1F,0x4B,0xBD,0x8B,0x8A,
    0x70,0x3E,0xB5,0x66,0x48,0x03,0xF6,0x0E,0x61,0x35,0x57,0xB9,0x86,0xC1,0x1D,0x9E,
    0xE1,0xF8,0x98,0x11,0x69,0xD9,0x8E,0x94,0x9B,0x1E,0x87,0xE9,0xCE,0x55,0x28,0xDF,
    0x8C,0xA1,0x89,0x0D,0xBF,0xE6,0x42,0x68,0x41,0x99,0x2D,0x0F,0xB0,0x54,0xBB,0x16,
]

hw = lambda b: bin(b).count('1')

# Show leakage model: HW of S-box output for each input
print("S-box output Hamming Weight (first 32 inputs):")
print("Input:  ", end="")
for i in range(32):
    print(f"{i:2d} ", end="")
print("\nS-box:  ", end="")
for i in range(32):
    print(f"{AES_SBOX[i]:02X} ", end="")
print("\nHW:     ", end="")
for i in range(32):
    print(f" {hw(AES_SBOX[i])} ", end="")
print("\n\nKey insight: HW varies significantly with input → enables CPA")

In [ ]:
# CPA Attack Implementation
print("=" * 60)
print("CPA ATTACK: KEY BYTE 0 RECOVERY")
print("=" * 60)

np.random.seed(42)

# Setup
n_traces = 500
n_samples = 100
secret_key = 0x2B  # Just byte 0 for this demonstration

# Generate simulated traces
plaintexts = np.random.randint(0, 256, n_traces)
traces = np.random.normal(0, 0.5, (n_traces, n_samples))

# Add leakage at time sample 50
for i in range(n_traces):
    sbox_out = AES_SBOX[plaintexts[i] ^ secret_key]
    traces[i, 50] += hw(sbox_out) * 0.4

# CPA attack
def pearson(x, y):
    n = len(x)
    mx, my = np.mean(x), np.mean(y)
    dx, dy = x - mx, y - my
    num = np.sum(dx * dy)
    den = np.sqrt(np.sum(dx**2) * np.sum(dy**2))
    return num / den if den > 0 else 0

correlations = np.zeros((256, n_samples))

for k_guess in range(256):
    intermediates = np.array([hw(AES_SBOX[pt ^ k_guess]) for pt in plaintexts], dtype=float)
    for t in range(n_samples):
        correlations[k_guess, t] = pearson(intermediates, traces[:, t])

# Find best key
max_corr = np.max(np.abs(correlations), axis=1)
best_key = np.argmax(max_corr)
rank = np.argsort(-max_corr).tolist().index(secret_key) + 1

print(f"Secret key byte: 0x{secret_key:02X}")
print(f"Recovered key:   0x{best_key:02X}")
print(f"Max correlation: {max_corr[best_key]:.4f}")
print(f"Key rank:        {rank}")
print(f"Attack:          {'SUCCESS' if best_key == secret_key else 'FAILED'}")

# Plot correlation heatmap
plt.figure(figsize=(14, 6))
plt.imshow(np.abs(correlations).T, aspect='auto', cmap='hot', origin='lower')
plt.colorbar(label='|Correlation|')
plt.axvline(x=secret_key, color='cyan', linestyle='--', linewidth=2, label=f'True key (0x{secret_key:02X})')
plt.title('CPA Correlation Heatmap (|ρ|)')
plt.xlabel('Key Guess')
plt.ylabel('Time Sample')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Full 128-bit Key Recovery
print("=" * 60)
print("FULL 128-BIT KEY RECOVERY (16 BYTES)")
print("=" * 60)

np.random.seed(42)

# Full 128-bit key
full_key = np.array([
    0x2B, 0x7E, 0x15, 0x16, 0x28, 0xAE, 0xD2, 0xA6,
    0xAB, 0xF7, 0x15, 0x88, 0x09, 0xCF, 0x4F, 0x3C
], dtype=np.uint8)

# Generate traces with leakage from all 16 bytes
n_traces = 1000
n_samples = 200
plaintexts_full = np.random.randint(0, 256, (n_traces, 16), dtype=np.uint8)
traces_full = np.random.normal(0, 0.3, (n_traces, n_samples))

# Add leakage for each key byte at different time offsets
for byte_idx in range(16):
    t_pos = 20 + byte_idx * 10  # Each byte leaks at different time
    for i in range(n_traces):
        sbox_out = AES_SBOX[plaintexts_full[i, byte_idx] ^ full_key[byte_idx]]
        traces_full[i, t_pos] += hw(sbox_out) * 0.5

# Recover all 16 key bytes
recovered_key = np.zeros(16, dtype=np.uint8)
all_max_corr = np.zeros(16)

for byte_idx in range(16):
    best_corr = -1
    for k_guess in range(256):
        intermediates = np.array([hw(AES_SBOX[plaintexts_full[i, byte_idx] ^ k_guess])
                                   for i in range(n_traces)], dtype=float)
        
        # Correlate with the expected time sample
        t_pos = 20 + byte_idx * 10
        corr = pearson(intermediates, traces_full[:, t_pos])
        
        if abs(corr) > abs(best_corr):
            best_corr = corr
            recovered_key[byte_idx] = k_guess
    
    all_max_corr[byte_idx] = abs(best_corr)
    status = '✓' if recovered_key[byte_idx] == full_key[byte_idx] else '✗'
    print(f"Byte {byte_idx:2d}: True=0x{full_key[byte_idx]:02X}  Recovered=0x{recovered_key[byte_idx]:02X}  "
          f"ρ={best_corr:.4f}  {status}")

print(f"\nFull key match: {'YES - ATTACK SUCCEEDED' if np.array_equal(recovered_key, full_key) else 'NO'}")
print(f"Average max correlation: {np.mean(all_max_corr):.4f}")
print(f"\nRecovered key: {[f'0x{b:02X}' for b in recovered_key]}")
print(f"Original key:  {[f'0x{b:02X}' for b in full_key]}")